# Setup

## Import libraries

In [15]:
import pandas as pd

## Create Globals for File Locations

In [16]:
gdpOriginalData = r"OriginalData/gdp-per-capita-worldbank.csv"
happinessOriginalData = r"OriginalData/WHR26_Data_Figure_2.1.xlsx"
exportLocation = r"ModifiedData/ExportedData.csv"

## Import both data sets

In [17]:
perCapitaDF = pd.read_csv(gdpOriginalData)
happinessDF = pd.read_excel(happinessOriginalData)


### Data Cleaning

#### Rename Country Column to match in both datasets

In [18]:

perCapitaDF = perCapitaDF.rename(columns={'Entity': 'Country name'})

#### Find unmatched countries in happpiness DF (the smaller dataset)

In [19]:
happyCountries = sorted(list(happinessDF["Country name"].unique()))
moneyCountries = sorted(list(perCapitaDF["Country name"].unique()))

matchedCountries = []
unmatchedCountries = []

for i in happyCountries:
    if i in moneyCountries:
        matchedCountries.append(i)
    else:
        unmatchedCountries.append(i)
unmatchedCountries

# with open("countries.txt", 'w') as f:
#     f.write(str(moneyCountries))

['Cuba',
 'Côte d’Ivoire',
 'DR Congo',
 'Hong Kong SAR of China',
 'Lao PDR',
 'North Cyprus',
 'Republic of Korea',
 'Republic of Moldova',
 'Russian Federation',
 'Somaliland Region',
 'South Sudan',
 'State of Palestine',
 'Swaziland',
 'Taiwan Province of China',
 'Türkiye',
 'Venezuela',
 'Viet Nam',
 'Yemen']

#### Rename unmatched countries to match Happiness dataset (if available)

In [20]:
# Not found: Cuba, North Cyprus, Somaliland Region, South Sudan, Taiwan, Venezuela, Yemen
correctionDictMoney = {
    "Cote d'Ivoire" : "Côte d’Ivoire",
    'Democratic Republic of Congo' : 'DR Congo',
    'Hong Kong' : "Hong Kong SAR of China",
    "Laos" : "Lao PDR",
    'South Korea' : "Republic of Korea",
    'Moldova' : "Republic of Moldova",
    'Russia' : "Russian Federation",
    'Palestine' : "State of Palestine",
    'Turkey' : "Türkiye",
    'Vietnam' : "Viet Nam"
}

# If any country names are incorrect/changed in the Happiness dataset, change here
correctionDictHappy = {
    "Swaziland" : "Eswatini"
}

perCapitaDF["Country name"] = perCapitaDF["Country name"].apply(lambda x: correctionDictMoney[x] if x in correctionDictMoney.keys() else x)
happinessDF["Country name"] = happinessDF["Country name"].apply(lambda x: correctionDictHappy[x] if x in correctionDictHappy.keys() else x)

#### Check remaining unmatched countries

In [21]:
happyCountries = sorted(list(happinessDF["Country name"].unique()))
moneyCountries = sorted(list(perCapitaDF["Country name"].unique()))

matchedCountries = []
unmatchedCountries = []

for i in happyCountries:
    if i in moneyCountries:
        matchedCountries.append(i)
    else:
        unmatchedCountries.append(i)
unmatchedCountries

['Cuba',
 'North Cyprus',
 'Somaliland Region',
 'South Sudan',
 'Taiwan Province of China',
 'Venezuela',
 'Yemen']

## Merge the two data sets

### \(worth 20%\)

In [22]:
# Place your code here.  Use more than one cell if needed.
mergedDF = pd.merge(happinessDF,perCapitaDF, how="inner", on=["Country name", "Year"], validate="1:1")
#This section was generated using Claude to simplify the reordering code
#-----
key_cols = ["Country name","Year", "Code",  'GDP per capita, PPP (constant 2021 international $)',"Life evaluation (3-year average)"]
other_cols = [c for c in mergedDF.columns if c not in key_cols]
mergedDF = mergedDF[key_cols + other_cols]
#-----
mergedDF

,Country name,Year,Code,"GDP per capita, PPP (constant 2021 international $)",Life evaluation (3-year average),Rank,Lower whisker,Upper whisker,Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,Dystopia + residual
0,Finland,2023,FIN,57063.5230,7.741,1,7.667,7.815,1.844,1.572,0.695,0.859,0.142,0.546,2.082
1,Denmark,2023,DNK,72097.3050,7.583,2,7.500,7.665,1.908,1.520,0.699,0.823,0.204,0.548,1.881
2,Iceland,2023,ISL,67255.6700,7.525,3,7.433,7.618,1.881,1.617,0.718,0.819,0.258,0.182,2.050
3,Sweden,2023,SWE,63114.6800,7.344,4,7.267,7.422,1.878,1.501,0.724,0.838,0.221,0.524,1.658
4,Israel,2023,ISR,48432.1100,7.341,5,7.277,7.405,1.803,1.513,0.740,0.641,0.153,0.193,2.298
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1760,Burundi,2011,BDI,951.1885,3.678,152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1761,Sierra Leone,2011,SLE,2471.7185,3.586,153,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1762,Central African Republic,2011,CAF,1517.1484,3.568,154,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1763,Benin,2011,BEN,2732.0125,3.493,155,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


A 1:1 validation is used to make sure that rows are unique to their country and year

## Verify the merge

#### Basic Checks:

##### Shape Check

In [23]:
mergedDF.shape

(1765, 15)

##### Years Covered

In [24]:
mergedDF["Year"].unique()

array([2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016, 2015, 2014, 2012,
       2011])

##### Nulls in important columns

In [25]:
# This check was suggested by AI 
mergedDF.isnull().sum()

Country name                                              0
Year                                                      0
Code                                                      0
GDP per capita, PPP (constant 2021 international $)       0
Life evaluation (3-year average)                          0
Rank                                                      0
Lower whisker                                          1056
Upper whisker                                          1056
Explained by: Log GDP per capita                       1059
Explained by: Social support                           1059
Explained by: Healthy life expectancy                  1060
Explained by: Freedom to make life choices             1059
Explained by: Generosity                               1059
Explained by: Perceptions of corruption                1059
Dystopia + residual                                    1060
dtype: int64

These checks confirm that the output dataframe is not malformed, range of years covered is limited to the data available in the Happiness report (as the GDP per capita data covers a significantly larger time period), and verifies that no important columns have any nulls (the last columns are used to break down the influences on a happiness score, but we will be doing our own analysis of only GDP per capita's effect)

#### Check how many countries overlapped

In [26]:
f"Result countries: {len(mergedDF["Country name"].unique())}, Original Happiness Countries: {len(happinessDF["Country name"].unique())}, Original GDP Countries: {len(perCapitaDF["Country name"].unique())} \nMissing countires explained by not being in GDP dataset?: {len(mergedDF["Country name"].unique()) == (len(happinessDF["Country name"].unique())-len(unmatchedCountries))}"

'Result countries: 160, Original Happiness Countries: 167, Original GDP Countries: 213 \nMissing countires explained by not being in GDP dataset?: True'

makes sure that no countries that are in both datasets are missing by comparing the number of countries in the merged dataframe with the number of countries in the happiness dataframe minus the countries that were not found while cleaning the GDP dataset, verifying that we are losing as little data as possible and not generating nonsensical data.

#### Check if there are any duplicate rows

In [27]:
"There are no duplicate rows" if len(mergedDF) == len(mergedDF.drop_duplicates()) else "There are duplicate rows"

'There are no duplicate rows'

Every rows should be be unique, as there are no duplicate entries for multiple countries in a single year

## Export merged data

In [28]:
# Place your code here.  Use more than one cell if needed.
mergedDF.to_csv(exportLocation)